# 05 — Rowhammer and DRAM Attacks

## What This Is
Rowhammer exploits DRAM physics: repeatedly accessing (hammering) a DRAM row causes charge to leak into adjacent rows, flipping bits without permission. TRRespass (2020) defeated Target Row Refresh mitigations in commercial DDR4.

## Why It Matters
Rowhammer has been used to escape browser sandboxes (drammer on Android), escalate privileges in Linux (by flipping a page table entry bit), and bypass cloud VM isolation. It affects hundreds of millions of deployed systems.

## Where the Community Stands
DDR5 introduces per-bank Refresh Management (RFM). LPDDR5 has on-die ECC. Software mitigations (KPTI, SMEP/SMAP) reduce exploitability but do not eliminate the underlying physics. DRAM vendors are still catching up.

## Physical Mechanism
DRAM cells are capacitors. Charge leaks over time (refreshed every 64ms). Hammering adjacent rows disturbs charge via electromagnetic coupling. The Target Row Refresh (TRR) mitigation tracks row activation counts — but modern patterns (Blacksmith non-uniform hammering) defeat TRR.

In [ ]:
import random
import math
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

# --- Simulated DRAM model ---

ROWS_PER_BANK    = 32768   # typical DDR4: 32K rows per bank
COLS_PER_ROW     = 1024    # 64-byte cache lines * 16
REFRESH_INTERVAL = 64      # ms
HAMMER_THRESHOLD = 140000  # activations per 64ms to cause bit flips (pre-TRR)
TRR_THRESHOLD    = 1000    # TRR triggers refresh above this count

@dataclass
class DRAMBank:
    n_rows: int
    activation_counts: Dict[int, int] = field(default_factory=dict)
    bit_flips: List[Tuple[int,int,str]] = field(default_factory=list)  # (row, col, direction)

    def activate(self, row: int) -> None:
        self.activation_counts[row] = self.activation_counts.get(row, 0) + 1

    def check_rowhammer(self, aggressor_rows: List[int],
                        with_trr: bool = False,
                        rng: random.Random = None) -> int:
        """Simulate rowhammer effect on victim rows adjacent to aggressors."""
        if rng is None:
            rng = random.Random(42)
        new_flips = 0
        for agg_row in aggressor_rows:
            count = self.activation_counts.get(agg_row, 0)
            effective = count
            if with_trr and count > TRR_THRESHOLD:
                # TRR reduces effective count for tracked aggressors
                effective = TRR_THRESHOLD
            if effective >= HAMMER_THRESHOLD:
                for victim in [agg_row - 1, agg_row + 1]:
                    if 0 <= victim < self.n_rows:
                        # Bit flip probability proportional to excess activations
                        excess = effective - HAMMER_THRESHOLD
                        flip_prob = min(1.0, excess / 50000)
                        if rng.random() < flip_prob:
                            col       = rng.randint(0, COLS_PER_ROW - 1)
                            direction = '0->1' if rng.random() > 0.5 else '1->0'
                            self.bit_flips.append((victim, col, direction))
                            new_flips += 1
        return new_flips

print(f'DRAM parameters:')
print(f'  Rows per bank:       {ROWS_PER_BANK:,}')
print(f'  Hammer threshold:    {HAMMER_THRESHOLD:,} activations')
print(f'  TRR threshold:       {TRR_THRESHOLD:,} activations')

In [ ]:
# Simulate single-sided hammering
rng = random.Random(42)
bank = DRAMBank(n_rows=ROWS_PER_BANK)

# Hammer row 1000 repeatedly
AGGRESSOR_ROW = 1000
N_ACTIVATIONS = 160000  # 160K > threshold

for _ in range(N_ACTIVATIONS):
    bank.activate(AGGRESSOR_ROW)

flips = bank.check_rowhammer([AGGRESSOR_ROW], with_trr=False, rng=rng)
print(f'Single-sided hammer (no TRR):')
print(f'  Aggressor row:   {AGGRESSOR_ROW}')
print(f'  Activations:     {N_ACTIVATIONS:,}')
print(f'  Bit flips:       {flips}')
if bank.bit_flips:
    for r, c, d in bank.bit_flips[:3]:
        print(f'  Flip at row={r} col={c}: {d}')

# With TRR
bank2 = DRAMBank(n_rows=ROWS_PER_BANK)
for _ in range(N_ACTIVATIONS):
    bank2.activate(AGGRESSOR_ROW)
flips2 = bank2.check_rowhammer([AGGRESSOR_ROW], with_trr=True, rng=rng)
print(f'\nSingle-sided hammer (with TRR):')
print(f'  Bit flips:       {flips2} (TRR mitigated)')

In [ ]:
# Double-sided and Blacksmith-style patterns
def double_sided_hammer(bank: DRAMBank, target_row: int,
                         n_activations: int, rng: random.Random) -> int:
    """Hammer both adjacent rows to target — doubles disturbance."""
    agg1 = max(0, target_row - 1)
    agg2 = min(bank.n_rows - 1, target_row + 1)
    for _ in range(n_activations):
        bank.activate(agg1)
        bank.activate(agg2)
    return bank.check_rowhammer([agg1, agg2], with_trr=False, rng=rng)

bank3 = DRAMBank(n_rows=ROWS_PER_BANK)
TARGET = 2000
flips3 = double_sided_hammer(bank3, TARGET, 100000, rng)
print(f'Double-sided hammer (target row {TARGET}):')
print(f'  Activations per aggressor: 100,000')
print(f'  Bit flips in target:       {flips3}')

# Blacksmith: non-uniform pattern to bypass TRR
def blacksmith_hammer(bank: DRAMBank, target: int, pattern: List[Tuple[int,int]],
                      n_repeats: int, rng: random.Random) -> int:
    """Pattern = [(row, n_activations)] per phase."""
    for _ in range(n_repeats):
        for row, acts in pattern:
            for __ in range(acts):
                bank.activate(row)
    aggressors = [r for r, _ in pattern]
    return bank.check_rowhammer(aggressors, with_trr=True, rng=rng)

bank4 = DRAMBank(n_rows=ROWS_PER_BANK)
# Uneven pattern: high on one side, low on other — defeats TRR tracking
pattern = [(TARGET-1, 900), (TARGET+1, 200), (TARGET-1, 900), (TARGET+2, 100)]
flips4  = blacksmith_hammer(bank4, TARGET, pattern, 200, rng)
print(f'\nBlacksmith non-uniform pattern (TRR enabled):')
print(f'  Pattern: {pattern}')
print(f'  Bit flips: {flips4}')

## Security Implications and Mitigations
A single bit flip in a page table entry can escalate an unprivileged process to kernel read/write access. Mitigations require both hardware (DDR5 RFM, on-die ECC) and software (KPTI, memory allocation isolation).

In [ ]:
# Page table exploit model
PAGE_TABLE_FLAGS = {
    'present':   0b00000001,   # bit 0
    'writable':  0b00000010,   # bit 1
    'user':      0b00000100,   # bit 2
    'nx':        1 << 63,      # NX bit
}

def pte_to_flags(pte: int) -> List[str]:
    flags = []
    if pte & PAGE_TABLE_FLAGS['present']:  flags.append('Present')
    if pte & PAGE_TABLE_FLAGS['writable']: flags.append('Writable')
    if pte & PAGE_TABLE_FLAGS['user']:     flags.append('User')
    if not (pte & PAGE_TABLE_FLAGS['nx']): flags.append('Executable')
    return flags

# A kernel page PTE: present, not user-accessible, not writable by user
kernel_pte = PAGE_TABLE_FLAGS['present']  # only present, supervisor-only
print(f'Original kernel PTE: {bin(kernel_pte)}')
print(f'Flags: {pte_to_flags(kernel_pte)}')

# Rowhammer bit flip: bit 2 flipped (user bit set!)
flipped_pte = kernel_pte ^ PAGE_TABLE_FLAGS['user']
print(f'\nAfter bit flip (user bit): {bin(flipped_pte)}')
print(f'Flags: {pte_to_flags(flipped_pte)}')
print(f'  *** User process now has access to kernel page! ***')

print('\nMitigations:')
mitigations = [
    ('DDR5 Refresh Management (RFM)',  'Hardware', 'Mandatory refresh on high-activation rows'),
    ('On-Die ECC (LPDDR5)',            'Hardware', 'Corrects 1-bit errors before CPU sees them'),
    ('Target Row Refresh (TRR)',        'Hardware', 'DRAM refresh adjacent rows — beatable by Blacksmith'),
    ('KPTI (Kernel Page Table Iso)',    'Software', 'Separate user/kernel page tables'),
    ('SMEP/SMAP',                      'Software', 'Prevent kernel executing/accessing user pages'),
    ('Guard pages',                    'Software', 'Non-mappable pages prevent cross-page hammering'),
]
for name, mtype, desc in mitigations:
    print(f'  [{mtype:<8}] {name:<35} {desc}')